<a href="https://colab.research.google.com/github/ShravaniPathak/edgarlytics/blob/main/Notebooks/data_preprocessing_embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install sentence-transformers tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_PATH = "/content/drive/MyDrive/sec-risk-analyzer"

DATA_PATH = os.path.join(BASE_PATH, "data/processed/risk_data.json")
EMBED_PATH = os.path.join(BASE_PATH, "data/embeddings")
CHUNK_PATH = os.path.join(BASE_PATH, "data/processed/chunks")

os.makedirs(EMBED_PATH, exist_ok=True)
os.makedirs(CHUNK_PATH, exist_ok=True)

print("Paths ready!")

Paths ready!


In [4]:
import json

with open(DATA_PATH, "r") as f:
    data = json.load(f)

print("Total filings loaded:", len(data))

Total filings loaded: 15


In [5]:
import re

def preprocess_text(text):
    text = re.sub(r"<.*?>", " ", text)  # remove HTML
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [6]:
def chunk_text(text, min_length=100, max_length=800):
    paragraphs = re.split(r"\n{2,}", text)

    chunks = []
    buffer = ""

    for p in paragraphs:
        p = p.strip()
        if not p:
            continue

        if len(buffer) + len(p) < max_length:
            buffer += " " + p
        else:
            if len(buffer) > min_length:
                chunks.append(buffer.strip())
            buffer = p

    if len(buffer) > min_length:
        chunks.append(buffer.strip())

    return chunks

In [7]:
from collections import defaultdict
from tqdm import tqdm

def prepare_chunks_with_metadata(results):
    all_chunks = []
    metadata = []

    year_chunks = defaultdict(list)
    company_chunks = defaultdict(list)

    for item in tqdm(results):
        ticker = item["ticker"]
        year = item["year"]

        text = preprocess_text(item["filing"])
        chunks = chunk_text(text)

        for chunk in chunks:
            all_chunks.append(chunk)

            metadata.append({
                "ticker": ticker,
                "year": year
            })

            year_chunks[year].append(chunk)
            company_chunks[ticker].append(chunk)

    return {
        "all_chunks": all_chunks,
        "metadata": metadata,
        "year_chunks": dict(year_chunks),
        "company_chunks": dict(company_chunks)
    }

In [8]:
processed_data = prepare_chunks_with_metadata(data)

all_chunks = processed_data["all_chunks"]
metadata = processed_data["metadata"]
year_chunks = processed_data["year_chunks"]
company_chunks = processed_data["company_chunks"]

print("Total chunks:", len(all_chunks))

100%|██████████| 15/15 [00:00<00:00, 121.24it/s]

Total chunks: 1649


In [9]:
print("\nChunks per year:")
for y in sorted(year_chunks.keys(), reverse=True):
    print(y, "→", len(year_chunks[y]))

print("\nChunks per company:")
for c in company_chunks:
    print(c, "→", len(company_chunks[c]))


Chunks per year:
2024 → 354
2023 → 330
2022 → 338
2021 → 327
2020 → 300

Chunks per company:
AAPL → 447
META → 889
AMZN → 313


In [10]:
with open(os.path.join(CHUNK_PATH, "all_chunks.json"), "w") as f:
    json.dump({
        "chunks": all_chunks,
        "metadata": metadata
    }, f)

print("Saved flat chunks + metadata")

Saved flat chunks + metadata


In [11]:
with open(os.path.join(CHUNK_PATH, "year_chunks.json"), "w") as f:
    json.dump(year_chunks, f)

print("Saved year-wise chunks")

Saved year-wise chunks


In [12]:
with open(os.path.join(CHUNK_PATH, "company_chunks.json"), "w") as f:
    json.dump(company_chunks, f)

print("Saved company-wise chunks")

Saved company-wise chunks


In [13]:
from tqdm import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer

batch_size = 32
embeddings_list = []
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

for i in tqdm(range(0, len(all_chunks), batch_size)):
    batch = all_chunks[i:i+batch_size]
    emb = model.encode(batch, show_progress_bar=False)
    embeddings_list.append(emb)

embeddings = np.vstack(embeddings_list)

print("Embeddings shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

100%|██████████| 52/52 [00:37<00:00,  1.39it/s]

Embeddings shape: (1649, 768)


In [14]:
import numpy as np
embed_file = os.path.join(EMBED_PATH, "all_embeddings.npy")
np.save(embed_file, embeddings)
print("Embeddings saved!")

Embeddings saved!


In [15]:
with open(os.path.join(EMBED_PATH, "metadata.json"), "w") as f:
    json.dump(metadata, f)

print("Metadata saved!")

Metadata saved!


In [16]:
from collections import defaultdict

year_index = defaultdict(list)
company_index = defaultdict(list)

for i, m in enumerate(metadata):
    year_index[m["year"]].append(i)
    company_index[m["ticker"]].append(i)

with open(os.path.join(EMBED_PATH, "year_index.json"), "w") as f:
    json.dump(year_index, f)

with open(os.path.join(EMBED_PATH, "company_index.json"), "w") as f:
    json.dump(company_index, f)

print("Index mappings saved!")

Index mappings saved!


In [17]:
print("Sample chunk:\n", all_chunks[0][:300])
print("\nEmbedding sample:\n", embeddings[0][:10])
print("\nMetadata sample:\n", metadata[0])

Sample chunk:
 Item 1A.    Risk Factors The Company’s business, reputation, results of operations, financial condition and stock price can be affected by a number of factors, whether currently known or unknown, including those described below. When any one or more of these risks materialize from time to time, the 

Embedding sample:
 [-0.03347567 -0.05158363 -0.01644519 -0.00668988  0.01546968  0.04703681
 -0.00404     0.01805403 -0.01910089 -0.0130867 ]

Metadata sample:
 {'ticker': 'AAPL', 'year': 2024}
